In [ ]:
# AT4-KN01 (spec v1.2 "Agent test 4 (scaled): unseen documents with gold answers, design before running", 2026-11-20):
# the thorough-mode index (Docling parse/OCR, method1_tree T384 chunks, bge-m3 vectors) of the 100 agent-test-4 PDFs
# on a Kaggle GPU, because the laptop estimate is over 3 hours. No answers here (no API keys on Kaggle): unzip the
# output at the repo root, then run the batch command on the laptop, where every index is a cache hit.
# Needs: GPU T4, Internet on (clone + model download), dataset AT4-KD01 attached (data/agent_test4/questions.csv and
# data/agent_test4/pdfs/*.pdf). No secrets. These documents are in no split; none is a study document.
COMMIT = ""  # the commit that holds src/agent/test4.py and pdfs/agent_test4_picks.csv; set before running
assert COMMIT, "set COMMIT to the commit with the agent test 4 code and picks"
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git checkout -q {COMMIT} && git log --oneline -1 && git status --porcelain | head

In [ ]:
!grep -iE "^(stanza|langchain[a-z_-]*|google-genai|pdfplumber|pypdf|pypdfium2)==" /kaggle/working/repo/requirements.txt | tee /kaggle/working/extra_pins.txt
!pip install -q -r /kaggle/working/extra_pins.txt

In [ ]:
# Docling as the docling extra in pyproject.toml; embedder stack as the study .venv. The cache is reused on the laptop
# only if the installed Docling version matches (src/agent/index.cached_index), so the version is pinned.
!pip install -q docling==2.143.0 sentence-transformers==6.2.0 transformers==5.22.1
!pip freeze > /kaggle/working/at4-kn01_pip_freeze.txt
!grep -i -E "^(docling|torch|transformers|sentence-transformers|numpy|pandas)==" /kaggle/working/at4-kn01_pip_freeze.txt
!cd /kaggle/working/repo && python -c "import src.agent.test4, src.agent.ask; print('imports OK')"
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "turn the GPU on"

In [ ]:
# Copy AT4-KD01 into the clone at its repo paths. Every PDF must be in pdfs/agent_test4_picks.csv of this commit
# with the same SHA-256, and questions.csv must name exactly those PDFs; anything else stops the notebook.
import hashlib
import shutil
import zipfile
from pathlib import Path

import pandas as pd

REPO, INPUT = Path("/kaggle/working/repo"), Path("/kaggle/input")
questions_file = next(INPUT.rglob("questions.csv"))
bundle = questions_file.parent
picks = pd.read_csv(REPO / "pdfs" / "agent_test4_picks.csv")
questions = pd.read_csv(questions_file)
assert set(questions["pdf"]) == set(picks["pdf"]) and len(picks) == 100, "questions.csv and the picks differ"
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
for pick in picks.itertuples(index=False):
    source, target = bundle / "pdfs" / Path(pick.pdf).name, REPO / pick.pdf
    assert sha(source) == pick.sha256, pick.pdf
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
shutil.copyfile(questions_file, REPO / "data" / "agent_test4" / "questions.csv")
print(len(picks), "PDFs copied and checked |", picks.groupby("kind")["pages"].sum().to_dict(), "pages")

In [ ]:
# Every index, one document at a time; failures are recorded in the run record and left out of batch_questions.csv.
!cd /kaggle/working/repo && python -m src.agent.test4 index --cache-root data/agent_test4/cache --machine kaggle --source AT4-KN01

In [ ]:
# One zip of the cache, batch_questions.csv and the run record, at repo paths (unzip at the repo root on the laptop).
files = sorted(path for path in (REPO / "data" / "agent_test4" / "cache").rglob("*") if path.is_file())
files += [REPO / "data" / "agent_test4" / "batch_questions.csv"] + sorted((REPO / "runs").glob("A-AT4-KN01-*.json"))
with zipfile.ZipFile("/kaggle/working/at4-kn01_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(REPO))
    archive.write("/kaggle/working/at4-kn01_pip_freeze.txt", "data/external/kaggle_out/AT4-KN01/at4-kn01_pip_freeze.txt")
print(len(files), "files zipped | run records:", sorted(path.name for path in files if path.parent.name == "runs"))
!ls -la /kaggle/working